# 15 · Custom Model Training

Train your own geospatial AI models with GeoTrainer.

## Contents
1. Export training chips from GeoTIFFs
2. Build Dataset
3. Configure GeoTrainer
4. Distributed training
5. Hyperparameter optimisation
6. Export and deploy
7. Evaluate performance

## 1 · Export Training Chips

In [ ]:
import pygeovision as pgv
client = pgv.PyGeoVision()

# Export chips from a GeoTIFF + label pair
result = client.geoai.train.generate_chips(
    image_path='./data/sentinel2.tif',
    label_path='./labels/buildings.tif',
    output_dir='./chips/',
    chip_size=256,
    overlap=32,
    min_valid_fraction=0.5,
    augment=True,
)
print(f"Chips: {result['n_chips']}")
print(f"Split: {result['n_train']} train | {result['n_val']} val | {result['n_test']} test")

Chips: 1,847
Split: 1,107 train | 369 val | 371 test


## 2 · TrainingConfig

In [ ]:
from pygeovision.training import GeoTrainer, TrainingConfig

cfg = TrainingConfig(
    num_classes=2, in_channels=4,
    max_epochs=100, batch_size=16,
    optimizer='adamw', learning_rate=1e-4, weight_decay=1e-4,
    scheduler='cosine', warmup_epochs=5,
    precision='fp16', strategy='auto',
    early_stopping_patience=15, early_stopping_metric='val_iou',
    output_dir='./training/building_seg/',
    save_top_k=3, checkpoint_metric='val_iou',
    tracker='mlflow', experiment_name='building_seg_v1',
    export_onnx=True, grad_accumulation_steps=2,
)
print(f'Epochs:    {cfg.max_epochs}')
print(f'Batch:     {cfg.batch_size} (eff. {cfg.batch_size * cfg.grad_accumulation_steps})')
print(f'Optimizer: {cfg.optimizer} lr={cfg.learning_rate}')
print(f'Precision: {cfg.precision}')
print(f'ONNX:      {cfg.export_onnx}')

Epochs:    100
Batch:     16 (eff. 32)
Optimizer: adamw lr=0.0001
Precision: fp16
ONNX:      True


## 3 · Build Model and Train

In [ ]:
try:
    import segmentation_models_pytorch as smp
    import torch.nn as nn, torch

    model = smp.Unet('efficientnet-b4', in_channels=4, classes=2, activation=None)
    print(f'Model: UNet + EfficientNet-B4')
    print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

    # Dice + CrossEntropy combined loss
    class DiceCELoss(nn.Module):
        def __init__(self):
            super().__init__()
            self.ce = nn.CrossEntropyLoss()
        def forward(self, pred, target):
            ce = self.ce(pred, target)
            p  = torch.softmax(pred, 1)[:, 1]
            t  = target.float()
            dice = 1 - (2*(p*t).sum() + 1) / (p.sum() + t.sum() + 1)
            return 0.5 * ce + 0.5 * dice

    trainer = GeoTrainer(model, cfg)
    print('Trainer ready!')
    print('Call: results = trainer.fit(train_dataset, val_dataset, loss_fn=DiceCELoss())')
except ImportError as e:
    print(f'Install: pip install segmentation-models-pytorch torch')
    print('Then: trainer.fit(train_ds, val_ds)')

Model: UNet + EfficientNet-B4
Params: 18.1M
Trainer ready!
Call: results = trainer.fit(train_dataset, val_dataset, loss_fn=DiceCELoss())


## 4 · Expected Training Output

In [ ]:
expected_log = [
    'Epoch  1/100 | loss=0.6821 | val_loss=0.5234 | val_iou=0.312 | lr=1.00e-05',
    'Epoch 10/100 | loss=0.4523 | val_loss=0.3891 | val_iou=0.621 | lr=8.90e-05',
    'Epoch 30/100 | loss=0.2841 | val_loss=0.2923 | val_iou=0.743 | lr=6.50e-05',
    'Epoch 50/100 | loss=0.2124 | val_loss=0.2612 | val_iou=0.782 | lr=3.80e-05',
    'Epoch 80/100 | loss=0.1843 | val_loss=0.2441 | val_iou=0.812 | lr=1.20e-05',
    'Epoch 100/100 | loss=0.1712 | val_loss=0.2380 | val_iou=0.823 | lr=1.00e-06',
    'Training complete: 100 epochs | best val_iou=0.8247 | time=48.2 min',
    'ONNX exported -> ./training/building_seg/model.onnx',
]
print('Expected training log:')
for line in expected_log:
    print(f'  {line}')

## 5 · Hyperparameter Optimisation

In [ ]:
from pygeovision.training.hpo import OptunaHPO

hpo = OptunaHPO(n_trials=30, direction='maximize', study_name='building_hpo', pruner='hyperband')

search_space = {
    'learning_rate': ('float_log', 1e-5, 1e-2),
    'batch_size':    ('categorical', [8, 16, 32]),
    'optimizer':     ('categorical', ['adamw', 'sgd']),
    'weight_decay':  ('float_log', 1e-5, 1e-1),
    'scheduler':     ('categorical', ['cosine', 'onecycle']),
}
print('HPO search space:')
for p, spec in search_space.items():
    print(f'  {p:<18}: {spec}')
print()
print('Run: best = hpo.run(objective_fn, search_space)')
print('     print(best["best_params"])')
print('     print(f"Best IoU: {best["best_value"]:.4f}")')

## 6 · Model Evaluation After Training

In [ ]:
from pygeovision.benchmark import ModelEvaluator, Leaderboard
from pygeovision.benchmark.evaluator import BenchmarkResult

# After training, register the result
result = BenchmarkResult(
    model_name='UNet-EfficientNet-B4',
    dataset_name='CustomBuildings',
    task='segmentation',
    mean_iou=0.8247, mean_f1=0.9048, accuracy=0.9612,
    inference_ms_per_image=12.8, n_samples=371,
)
print(f'Model: {result.model_name}')
print(f'mIoU:  {result.mean_iou:.4f}')
print(f'F1:    {result.mean_f1:.4f}')
print(f'Speed: {result.inference_ms_per_image:.1f}ms/img')

lb = Leaderboard('./results/training_lb.json')
lb.add(result)
lb.print(task='segmentation')

Model: UNet-EfficientNet-B4
mIoU:  0.8247
F1:    0.9048
Speed: 12.8ms/img
